# ConsultaAI — Etapa 1.2: Instanciação, Inferência e Benchmark de Modelos

Este notebook carrega o dataset fatiado na Etapa 1.1, instancia os modelos de ASR (Wav2Vec2, Meta MMS e Whisper), executa a inferência e calcula as métricas de acurácia (WER/CER) em relação aos Gabaritos A (coloquial) e B (normalizado), além de medir tempos de execução.

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Adiciona a pasta src ao sys.path para importar os módulos locais
sys.path.append(str(Path("..").resolve()))

from src.models.inference import ASRModelWrapper
from src.evaluation.metrics import calculate_wer, calculate_cer

## 1. Carregamento dos Metadados Preprocessados

Carregamos o arquivo `metadata.csv` contendo a localização dos áudios fatiados e as referências textuais.

In [2]:
metadata_path = Path("../data/input/processed/metadata.csv")
df = pd.read_csv(metadata_path)
print(f"Total de segmentos disponíveis para benchmark: {len(df)}")
df.head()

Total de segmentos disponíveis para benchmark: 16963


,segment_id,file_id,speaker,start_ms,end_ms,audio_path,text_raw,text_gabarito_a,text_gabarito_b
0,besqau01_0000,besqau01,AND,28,2492,data/output/audio/besqau01_0000_AND.wav,e como é que tão as questão das vozes //,e como é que tão as questão das vozes,e como é que estão as questão das vozes
1,besqau01_0001,besqau01,GER,2869,7130,data/output/audio/besqau01_0001_GER.wav,é aquele negócio que eu falei né doutor / &he ...,é aquele negócio que eu falei né doutor o as v...,é aquele negócio que eu falei né doutor o as v...
2,besqau01_0002,besqau01,GER,7130,9101,data/output/audio/besqau01_0002_GER.wav,<indiferente> do remédio / é que o meu +,indiferente do remédio é que o meu,indiferente do remédio é que o meu
3,besqau01_0003,besqau01,AND,7246,7970,data/output/audio/besqau01_0003_AND.wav,<hum hum> //,hum hum,hum hum
4,besqau01_0004,besqau01,GER,9101,13188,data/output/audio/besqau01_0004_GER.wav,quando tomar o remédio / quando nũ tava / toma...,quando tomar o remédio quando nũ tava tomando ...,quando tomar o remédio quando não estava toman...


## 2. Seleção de Amostra para Avaliação Rápida

Como transcrever as 9.5 horas de áudio (16.963 segmentos) pode demorar bastante em CPU, vamos selecionar um subset para o benchmark inicial. 

Por padrão, usaremos os primeiros 100 segmentos do arquivo `besqau01`, mas você pode alterar para o tamanho de amostra que desejar.

In [3]:
# Filtra segmentos de besqau01 como amostra estável de teste (ou use df.sample(N))
df_sample = df[df["file_id"] == "besqau01"].head(100).copy()
print(f"Tamanho da amostra para o benchmark: {len(df_sample)} segmentos")

Tamanho da amostra para o benchmark: 100 segmentos


## 3. Avaliação do Modelo Whisper (base)

Instanciamos o Whisper Base via faster-whisper e rodamos a inferência na nossa amostra.

In [4]:
# Instancia o modelo Whisper Base via faster-whisper
whisper_base_model = ASRModelWrapper(model_type="whisper", model_id_or_size="base")

Carregando modelo whisper (base) no dispositivo: mps...
Modelo carregado com sucesso!


In [5]:
whisper_base_preds = []
whisper_base_times = []
audio_durations = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Whisper Base"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = whisper_base_model.transcribe(audio_path)
        whisper_base_preds.append(res["text"])
        whisper_base_times.append(res["execution_time_s"])
        audio_durations.append(res["audio_len_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        whisper_base_preds.append("")
        whisper_base_times.append(0.0)
        audio_durations.append(0.0)

df_sample["pred_whisper_base"] = whisper_base_preds
df_sample["time_whisper_base"] = whisper_base_times
df_sample["audio_len"] = audio_durations

Whisper Base:   0%|          | 0/100 [00:00<?, ?it/s]

## 4. Avaliação do Modelo Whisper (small)

Instanciamos o Whisper Small via faster-whisper e rodamos a inferência na nossa amostra.

In [6]:
# Instancia o modelo Whisper Small via faster-whisper
whisper_small_model = ASRModelWrapper(model_type="whisper", model_id_or_size="small")

Carregando modelo whisper (small) no dispositivo: mps...
Modelo carregado com sucesso!


In [7]:
whisper_small_preds = []
whisper_small_times = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Whisper Small"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = whisper_small_model.transcribe(audio_path)
        whisper_small_preds.append(res["text"])
        whisper_small_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        whisper_small_preds.append("")
        whisper_small_times.append(0.0)

df_sample["pred_whisper_small"] = whisper_small_preds
df_sample["time_whisper_small"] = whisper_small_times

Whisper Small:   0%|          | 0/100 [00:00<?, ?it/s]

## 5. Avaliação do Modelo Wav2Vec2 (Português)

Instanciamos o Wav2Vec2 treinado para o português (`jonatasgrosman/wav2vec2-large-xlsr-53-portuguese`) via Hugging Face e rodamos a inferência.

In [ ]:
# Instancia o modelo Wav2Vec2
wav2vec2_model = ASRModelWrapper(
    model_type="wav2vec2",
    model_id_or_size="jonatasgrosman/wav2vec2-large-xlsr-53-portuguese"
)

Carregando modelo wav2vec2 (jonatasgrosman/wav2vec2-large-xlsr-53-portuguese) no dispositivo: mps...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

[transformers] Could not load the `decoder` for jonatasgrosman/wav2vec2-large-xlsr-53-portuguese. Defaulting to raw CTC. Error: No module named 'kenlm'
[transformers] Try to install `kenlm`: `pip install kenlm
[transformers] Try to install `pyctcdecode`: `pip install pyctcdecode


Modelo carregado com sucesso!


In [9]:
wav2vec2_preds = []
wav2vec2_times = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Wav2Vec2 PT"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = wav2vec2_model.transcribe(audio_path)
        wav2vec2_preds.append(res["text"])
        wav2vec2_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        wav2vec2_preds.append("")
        wav2vec2_times.append(0.0)

df_sample["pred_wav2vec2_pt"] = wav2vec2_preds
df_sample["time_wav2vec2_pt"] = wav2vec2_times

Wav2Vec2 PT:   0%|          | 0/100 [00:00<?, ?it/s]

## 6. Avaliação do Modelo Meta MMS-1B (Português)

Instanciamos o Meta MMS-1B (`facebook/mms-1b-all`) via Hugging Face e rodamos a inferência.

In [ ]:
# Instancia o modelo Meta MMS-1B
mms_model = ASRModelWrapper(
    model_type="mms",
    model_id_or_size="facebook/mms-1b-all"
)

Carregando modelo mms (facebook/mms-1b-all) no dispositivo: mps...


model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.por.safetensors:   0%|          | 0.00/9.06M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Modelo carregado com sucesso!


In [11]:
mms_preds = []
mms_times = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Meta MMS-1B"):
    audio_path = Path("..") / "data/input/processed/audio" / Path(row["audio_path"]).name
    try:
        res = mms_model.transcribe(audio_path)
        mms_preds.append(res["text"])
        mms_times.append(res["execution_time_s"])
    except Exception as e:
        print(f"Erro no segmento {row['segment_id']}: {e}")
        mms_preds.append("")
        mms_times.append(0.0)

df_sample["pred_mms_1b"] = mms_preds
df_sample["time_mms_1b"] = mms_times

Meta MMS-1B:   0%|          | 0/100 [00:00<?, ?it/s]

## 7. Cálculo de Métricas Comparativas (WER/CER)

Calculamos as taxas de erro das predições de cada modelo comparando contra as referências dos Gabaritos A (coloquial) e B (normalizado).

In [12]:
# Filtra linhas válidas (onde a inferência funcionou)
df_valid = df_sample[df_sample["audio_len"] > 0].copy()

# Referências
ref_a = df_valid["text_gabarito_a"].fillna("").tolist()
ref_b = df_valid["text_gabarito_b"].fillna("").tolist()

# Predições
pred_whisper_base = df_valid["pred_whisper_base"].fillna("").tolist()
pred_whisper_small = df_valid["pred_whisper_small"].fillna("").tolist()
pred_wav2vec2 = df_valid["pred_wav2vec2_pt"].fillna("").tolist()
pred_mms = df_valid["pred_mms_1b"].fillna("").tolist()

# WER
wer_whisper_base_a = calculate_wer(ref_a, pred_whisper_base)
wer_whisper_base_b = calculate_wer(ref_b, pred_whisper_base)
wer_whisper_small_a = calculate_wer(ref_a, pred_whisper_small)
wer_whisper_small_b = calculate_wer(ref_b, pred_whisper_small)
wer_wav2vec2_a = calculate_wer(ref_a, pred_wav2vec2)
wer_wav2vec2_b = calculate_wer(ref_b, pred_wav2vec2)
wer_mms_a = calculate_wer(ref_a, pred_mms)
wer_mms_b = calculate_wer(ref_b, pred_mms)

# CER
cer_whisper_base_a = calculate_cer(ref_a, pred_whisper_base)
cer_whisper_base_b = calculate_cer(ref_b, pred_whisper_base)
cer_whisper_small_a = calculate_cer(ref_a, pred_whisper_small)
cer_whisper_small_b = calculate_cer(ref_b, pred_whisper_small)
cer_wav2vec2_a = calculate_cer(ref_a, pred_wav2vec2)
cer_wav2vec2_b = calculate_cer(ref_b, pred_wav2vec2)
cer_mms_a = calculate_cer(ref_a, pred_mms)
cer_mms_b = calculate_cer(ref_b, pred_mms)

# Métricas de Hardware
total_audio_s = df_valid["audio_len"].sum()
rtf_whisper_base = df_valid["time_whisper_base"].sum() / total_audio_s
rtf_whisper_small = df_valid["time_whisper_small"].sum() / total_audio_s
rtf_wav2vec2 = df_valid["time_wav2vec2_pt"].sum() / total_audio_s
rtf_mms = df_valid["time_mms_1b"].sum() / total_audio_s

# Montando o Report
results = {
    "Modelo": ["Whisper Base", "Whisper Small", "Wav2Vec2 PT (XLSR-53)", "Meta MMS-1B"],
    "WER - Gabarito A (Coloquial)": [wer_whisper_base_a, wer_whisper_small_a, wer_wav2vec2_a, wer_mms_a],
    "WER - Gabarito B (Normalizado)": [wer_whisper_base_b, wer_whisper_small_b, wer_wav2vec2_b, wer_mms_b],
    "CER - Gabarito A (Coloquial)": [cer_whisper_base_a, cer_whisper_small_a, cer_wav2vec2_a, cer_mms_a],
    "CER - Gabarito B (Normalizado)": [cer_whisper_base_b, cer_whisper_small_b, cer_wav2vec2_b, cer_mms_b],
    "RTF (Fator de Tempo Real)": [rtf_whisper_base, rtf_whisper_small, rtf_wav2vec2, rtf_mms],
    "Tempo Médio por Segmento (s)": [
        df_valid["time_whisper_base"].mean(),
        df_valid["time_whisper_small"].mean(),
        df_valid["time_wav2vec2_pt"].mean(),
        df_valid["time_mms_1b"].mean()
    ]
}

df_results = pd.DataFrame(results)
df_results

,Modelo,WER - Gabarito A (Coloquial),WER - Gabarito B (Normalizado),CER - Gabarito A (Coloquial),CER - Gabarito B (Normalizado),RTF (Fator de Tempo Real),Tempo Médio por Segmento (s)
0,Whisper Base,0.721854,0.687086,0.443631,0.427969,0.726742,1.467560
1,Whisper Small,0.541391,0.496689,0.321010,0.305692,1.060736,2.142019
2,Wav2Vec2 PT (XLSR-53),0.766556,0.756623,0.443997,0.443781,0.055197,0.111463
3,Meta MMS-1B,0.693709,0.688742,0.371523,0.380183,0.108751,0.219608


## 8. Análise de Amostras Lado a Lado

Vamos visualizar algumas predições reais comparadas com as referências originais para entender as características e tipos de erros de cada modelo.

In [13]:
columns_to_show = [
    "speaker",
    "text_gabarito_a",
    "text_gabarito_b",
    "pred_whisper_base",
    "pred_whisper_small",
    "pred_wav2vec2_pt",
    "pred_mms_1b"
]
df_valid[columns_to_show].head(10)

,speaker,text_gabarito_a,text_gabarito_b,pred_whisper_base,pred_whisper_small,pred_wav2vec2_pt,pred_mms_1b
0,AND,e como é que tão as questão das vozes,e como é que estão as questão das vozes,Como é que estão os questões das vozes?,E como é que estão as questões das vozes?,e comectose que estão das vozes,e comeque tão s questão das vozes
1,GER,é aquele negócio que eu falei né doutor o as v...,é aquele negócio que eu falei né doutor o as v...,"Agora vamos fazendo a dor, a voz é ouço mesmo","Agora eu vou falar no doutor, as vozes eu ouço...",andam agora o falendo doator e a a volta ou ou...,ela tem agora com calilde doutor as vós eu os...
2,GER,indiferente do remédio é que o meu,indiferente do remédio é que o meu,e o ferido é meio de comer.,O diferente do remédio é com o meu...,um diferendo remédio comelo,undiferente do remedioco com me
3,AND,hum hum,hum hum,Um inferno.,Inferno,me diferedo,difereo
4,GER,quando tomar o remédio quando nũ tava tomando ...,quando tomar o remédio quando não estava toman...,"quando está morrer mesmo, quando está lá toman...","Quando eu tomar o remédio, quando eu to tomand...",quanto talmar remédio cando talna talaman as o...,quanto tomao remédio quando toma tão humano a ...
5,GER,mas nũ me prejudica em nada que eu nũ dou faço...,mas não me prejudica em nada que eu não dou fa...,Mas não me prejudica em nada que eu não dou. F...,"mas não me prejudica em nada, que eu não faço ...",mas nme peodica em nádica andoa fasço nada é r...,ma n mi perjudika é nada ka nun dove fas nada ...
6,AND,mas ela fica falando p' cê fazer coisa errada,mas ela fica falando para você fazer coisa errada,Mas ela fica falando se faz algo de errado.,Mas ela fica falando se fazer coisa errada.,mas ela fica falando ser fazer coiso errado,mas ela fica falando sa fazer coisa érravo
7,GER,não fala não,não fala não,Não. Fala não.,"Não, fala não.",não fala não,naofano
8,GER,senão o diabo fala né,senão o diabo fala né,"Então, já eu falo, né?","São diabos falo, né?",so o diabo fala né,são diabo fala ne
9,AND,então mas o diabo tem aparecido,então mas o diabo tem aparecido,"Então, mas já tem aparecido.","Então, mas já tem parecido?",tomas o diabo tempopareciado,então mas o diabo temh parecido


## 9. Salvamento do Dataset de Saída Comparativo

Salvamos as predições completas da amostra em formato CSV para visualização dos resultados


In [14]:
output_csv_path = Path("../data/output/benchmark_predictions.csv")
df_valid.to_csv(output_csv_path, index=False, encoding="utf-8")
print(f"Predições detalhadas salvas com sucesso em:\n{output_csv_path.resolve()}")

Predições detalhadas salvas com sucesso em:
/Users/thomasdemello/Documents/Mestrado/ConsultaAI/INF2475-DeepLearning/data/output/benchmark_predictions.csv
